In [2]:
from rag_helper import RAGBase

In [3]:
from sqlitesearch import TextSearchIndex

sqlliteindex = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from openai import OpenAI
from ingest import load_faq_data

In [5]:

import os
documents = load_faq_data()

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

assistant = RAGBase(
    index=sqlliteindex,
    llm_client=openai_client,
    model="openai/gpt-oss-20b"
)

answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

Yes—you can still join the course. Just keep in mind that if you want to earn a certificate, you’ll need to submit your capstone project while the submission window is still open.


In [11]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return sqlliteindex.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [24]:
search("I just discovered the course. Can I join it?")

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': 'aa310de435',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Can I run the course locally instead of Codespaces?',
  'answer': 'Yes. Codespaces is just the easiest way for everyone to s

In [13]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [ ]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="openai/gpt-oss-120b",
    input=messages,
    tools=[search_tool]

)


In [88]:
response.output[1]

ResponseFunctionToolCall(arguments='{"query":"enrollment eligibility course join"}', call_id='fc_9cf0f425-91cc-4a48-8c2c-4db270fec022', name='search', type='function_call', id='fc_9cf0f425-91cc-4a48-8c2c-4db270fec022', caller=None, namespace=None, status='completed')

In [86]:
import json

call=response.output[1]
args = json.loads(response.output[1].arguments)
response.output[1].name

results = search(**args)
result_json = json.dumps(results, indent=2)

In [85]:
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "bd31146b0e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "When will the course be offered next?",
    "answer": "Summer 2027."
  },
  {
    "id": "69d122f12e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",
    "answer": "No, you can only get a certificate if you finish the course with a \"live\" cohort.\n\nTo get the certificate, you need to finish a capstone project and complete the\nrequired peer reviews. Homework is not required. You can work through the\nmaterial a

In [87]:
{
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
}

{'type': 'function_call_output',
 'call_id': 'fc_9cf0f425-91cc-4a48-8c2c-4db270fec022',
 'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "bd31146b0e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "When will the course be offered next?",\n    "answer": "Summer 2027."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",\n    "answer": "No, you can only get a certificate if you finish the course with a \\"live\\" cohort.\\n\\nTo get the certificate, you need to fini

In [89]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [90]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseReasoningItem(id='resp_01ky9trjz0f269npynckgdckb4', summary=[], type='reasoning', content=[Content(text='The user asks: "I just discovered the course. Can I join it?" We need to answer according to policy. Probably they are asking about enrollment. We need to explain enrollment criteria: must be a student in a University of Washington program and a current student in the Graduate School. Possibly also about the "no cost" but enrollment limited. Provide steps: go to enrollment page, sign up, meet prerequisites, maybe contact instructor.\n\nWe also may want to check FAQ database for any relevant entries about eligibility. Use function search.', type='reasoning_text')], encrypted_content=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"enrollment eligibility course join"}', call_id='fc_9cf0f425-91cc-4a48-8c2c-4db270fec022', name='search', type='function_call', id='fc_9cf0f425-91

In [91]:
response = openai_client.responses.create(
    model="openai/gpt-oss-120b",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes—you can join the LLM\u202fZoomcamp even if you just discovered it now. The course is open‑ended, so you can start working through the videos, notebooks, and assignments right away.\n\n**A couple of things to keep in mind:**\n\n| Goal | What you need to do |\n|------|---------------------|\n| **Just learn the material** | No registration or deadline is required. Feel free to dive into the docs, videos, and GitHub repo whenever you like. |\n| **Earn a certificate** | You must **submit a capstone project and complete the required peer reviews while the live cohort is still accepting submissions**. After that window closes, new participants can still learn, but a certificate will no longer be available. |\n\nSo go ahead and start the course today; if you aim for the certificate, plan to finish and submit your project before the current submission deadline. Happy learning!'

In [92]:
print(response.output_text)

Yes—you can join the LLM Zoomcamp even if you just discovered it now. The course is open‑ended, so you can start working through the videos, notebooks, and assignments right away.

**A couple of things to keep in mind:**

| Goal | What you need to do |
|------|---------------------|
| **Just learn the material** | No registration or deadline is required. Feel free to dive into the docs, videos, and GitHub repo whenever you like. |
| **Earn a certificate** | You must **submit a capstone project and complete the required peer reviews while the live cohort is still accepting submissions**. After that window closes, new participants can still learn, but a certificate will no longer be available. |

So go ahead and start the course today; if you aim for the certificate, plan to finish and submit your project before the current submission deadline. Happy learning!


In [6]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = "I just discovered the course. Can I join it?"

In [7]:
messages=[
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question}
]

In [119]:
response = openai_client.responses.create(
    model="openai/gpt-oss-120b",
    input=messages,
    tools=[search_tool],
)

In [120]:
response.output

[ResponseReasoningItem(id='resp_01kya0hy68ft7vb44g6ew3j3mr', summary=[], type='reasoning', content=[Content(text='User asks: "I just discovered the course. Can I join it?" Need to answer about enrollment, prerequisites, registration process, maybe open enrollment? Use FAQ search.', type='reasoning_text')], encrypted_content=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"can I join the course enrollment prerequisites"}', call_id='fc_c7e6ee6c-225b-424e-aec6-488a68bd727b', name='search', type='function_call', id='fc_c7e6ee6c-225b-424e-aec6-488a68bd727b', caller=None, namespace=None, status='completed')]

In [121]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [125]:
messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

ASSISTANT:
**Yes – you can still join the LLM Zoomcamp.**  

- **No formal registration is required** – you can start working through the notebooks, homework, and videos right away.  
- **If you want a certificate**, you need to submit a capstone project (and complete the required peer‑reviews) while a live cohort is still open for submissions. After the deadline for that cohort closes, new projects won’t be accepted for certification.  

So feel free to dive in now; just keep an eye on the cohort schedule if you plan to earn the certificate.

---

*Is there anything else you’d like to know about enrollment, the cohort timeline, or the certification process?*


In [126]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseReasoningItem(id='resp_01kya0hy68ft7vb44g6ew3j3mr', summary=[], type='reasoning', content=[Content(text='User asks: "I just discovered the course. Can I join it?" Need to answer about enrollment, prerequisites, registration process, maybe open enrollment? Use FAQ search.', type='reasoning_text')], encrypted_content=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"can I join the co

In [124]:
response = openai_client.responses.create(
    model="openai/gpt-oss-120b",
    input=messages,
    tools=[search_tool],
)
response.output

[ResponseReasoningItem(id='resp_01kya0kx1mfkavnfhw5w5kf5be', summary=[], type='reasoning', content=[Content(text='We have the FAQ entry that exactly answers: "I just discovered the course. Can I still join?" Answer: Yes, but need to submit project while still accepting submissions for certificate.\n\nThus respond accordingly, include details about enrollment, registration optional, can start learning, but certificate requires project submission during live cohort.\n\nAsk if they need more info.', type='reasoning_text')], encrypted_content=None, status='completed'),
 ResponseOutputMessage(id='msg_01kya0kx1mfkb9xrxk5dj79acs', content=[ResponseOutputText(annotations=[], text='**Yes – you can still join the LLM\u202fZoomcamp.**  \n\n- **No formal registration is required** – you can start working through the notebooks, homework, and videos right away.  \n- **If you want a certificate**, you need to submit a capstone project (and complete the required peer‑reviews) while a live cohort is st

In [139]:
def agent_loop(instructions, question, model="openai/gpt-oss-120b") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it=1
    while(True):
        has_function_calls = False
        print(f"iteration #{it}...")

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool],
        )

        messages.extend(response.output)
        has_function_calls = False

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)
        it = it + 1
        if has_function_calls == False:
            break
    return last_answer

In [140]:
agent_loop(instructions, question, model="openai/gpt-oss-120b")

iteration #1...
function_call: search {"query":"join course enrollment eligibility"}
iteration #2...
ASSISTANT:
Absolutely! You can join the LLM Zoomcamp even if you just discovered it now. The course is open‑ended: you can start watching the videos, working through the notebooks, and submitting homework right away—there’s no “closed‑door” enrollment list.

A couple of quick things to keep in mind:

| What you can do now | What you need for a certificate |
|---------------------|----------------------------------|
| **Start learning immediately** – all videos, notebooks, and the GitHub repo are publicly available. | **Finish a capstone project and complete the required peer‑reviews** during a live cohort’s submission window. |
| **Submit homework** while the form is still open (the platform accepts submissions from anyone). | **Certificate** is only awarded to participants who complete the project and peer‑review during a live cohort; you can prepare the project in self‑paced mode, but

'Absolutely!\u202fYou can join the LLM\u202fZoomcamp even if you just discovered it now. The course is open‑ended: you can start watching the videos, working through the notebooks, and submitting homework right away—there’s no “closed‑door” enrollment list.\n\nA couple of quick things to keep in mind:\n\n| What you can do now | What you need for a certificate |\n|---------------------|----------------------------------|\n| **Start learning immediately** – all videos, notebooks, and the GitHub repo are publicly available. | **Finish a capstone project and complete the required peer‑reviews** during a live cohort’s submission window. |\n| **Submit homework** while the form is still open (the platform accepts submissions from anyone). | **Certificate** is only awarded to participants who complete the project and peer‑review during a live cohort; you can prepare the project in self‑paced mode, but the final submission must happen while a live cohort is accepting it. |\n\n**Next steps**\n\n

In [8]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [14]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [15]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return sqlliteindex.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [16]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [17]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [18]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(client=openai_client,model="openai/gpt-oss-120b")
)

In [19]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


OS,Command / Action
macOS,Download the .pkg from https://ollama.com/download and double‑click to install.
Windows,Download the .msi from the same page and run the installer.
Linux,Run the one‑liner in a terminal: curl -fsSL https://ollama.com/install.sh \| sh
Docker (optional),If you prefer containerised execution: docker run -d -p 11434:11434 --name ollama ollama/ollama
Action,Command
List cached models,ollama list
Delete a model (free space),ollama rm llama3
Stop the server (if you started it manually),pkill -f ollama (Unix) or stop the Docker container
Change the listening port,OLLAMA_HOST=0.0.0.0 OLLAMA_PORT=12345 ollama serve


/workspaces/llm-zoomcamp/.venv/lib/python3.12/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'openai/gpt-oss-120b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


In [20]:
result.cost

In [21]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseReasoningItem(id='resp_01kya8ss26f6vbhgdrxh7vwh3w', summary=[], type='reasoning', content=[Content(text='The user asks: "How do I run Olama locally?" They likely refer to a software project named "Olama". Need to search.', type='reasoning_text')], encrypted_content=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"Olama run locally"}',

In [22]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


In [23]:
runner.run()

-> Response received


Chat ended.


/workspaces/llm-zoomcamp/.venv/lib/python3.12/site-packages/toyaikit/chat/runners.py:184: UnknownModelWarning: No pricing data for model 'openai/gpt-oss-120b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  combined_cost = self.pricing_config.calculate_cost(


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None), EasyInputMessage(content='hi', role='user', phase=None, type=None), ResponseReasoningItem(id='resp_01kya8z8shfzc9evv570y74fap', summary=[], type='reasoning', content=[Content(text='User just says "hi". As a teaching assistant, respond friendly and ask how can help.', type='reasoning_text')], encrypted_content=None, status='completed'), ResponseOutputMessage(id='msg_01kya8z8shfzct45j5ckbn52ek', content=[ResponseOutputText(annotations=[],